# 🏆 R2AI2026 - ViFinQA Text-to-Pandas Pipeline
**Google Colab Free (T4 GPU - 15GB VRAM)**

## Nguyên tắc vàng:
- **Code** → Ổ SSD `/content/Code_Moi` (clone từ GitHub)
- **Data gốc** (ViFinQA) → Google Drive (chỉ đọc)
- **Data sinh ra** (CSV/Metadata/Index) → Ổ SSD `/content/data_output` (tốc độ cao)
- **Kết quả** (submission) → Google Drive (lưu trữ lâu dài)

---
### Nick chính (chạy Pipeline 1 lần đầu):
Chạy: Ô 1 → Ô 2 → Ô 3 → **Ô 4** → **Ô 5** → (Ô 6 → Ô 7 nếu muốn luôn Pipeline 2)

### Nick phụ (chỉ chạy Pipeline 2):
Chạy: Ô 1 → Ô 2 → Ô 3 → Bỏ qua Ô 4 → Bỏ qua Ô 5 → **Ô 5b** → **Ô 6** → Ô 7

In [ ]:
# ============================================================
# Ô 1: MOUNT DRIVE + CLONE CODE VÀO SSD
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# --- Clone code vào SSD ---
CODE_DIR = '/content/Code_Moi'
if os.path.exists(CODE_DIR):
    %cd {CODE_DIR}
    !git pull
else:
    %cd /content
    !git clone https://github.com/HoangKhang226/AI-Financial-Data-Assistant.git Code_Moi
    %cd {CODE_DIR}

print(f'\n\u2705 Code: {os.getcwd()}')

In [ ]:
# ============================================================
# Ô 2: CÀI THƯ VIỆN
# ============================================================
!pip uninstall -y torchaudio torchvision 2>/dev/null
!pip install -r requirements.txt -q
print('\n\u2705 Cài đặt hoàn tất!')

In [ ]:
# ============================================================
# Ô 3: KIỂM TRA GPU
# ============================================================
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('\u274c KHÔNG CÓ GPU! Vào Runtime > Change runtime type > T4 GPU')

import transformers; print(f'Transformers: {transformers.__version__}')
import vllm; print(f'vLLM: {vllm.__version__}')
print('\n\u2705 Môi trường sẵn sàng!')

---
## 📦 PIPELINE 1 (Nick chính - chỉ chạy 1 lần)

In [ ]:
# ============================================================
# Ô 4: PIPELINE 1 - INGESTION (đọc Drive, ghi SSD)
# CHỈ CHẠY 1 LẦN TRÊN NICK CHÍNH
# ============================================================
import os

SSD_DATA = '/content/data_output'
os.makedirs(f'{SSD_DATA}/csv_warehouse', exist_ok=True)
os.makedirs(f'{SSD_DATA}/metadata', exist_ok=True)
os.makedirs(f'{SSD_DATA}/index', exist_ok=True)

DRIVE_DATA = '/content/drive/MyDrive/Colab Notebooks/Text2Pandas/data'
INPUT_DIR = os.path.join(DRIVE_DATA, 'ViFinQA/financial_statements')

!python -m src.pipeline_1_ingestion.pipeline \
    --input "{INPUT_DIR}" \
    --csv-out "{SSD_DATA}/csv_warehouse" \
    --meta-out "{SSD_DATA}/metadata" \
    --summ-out "{SSD_DATA}/index/summaries.jsonl"

n_csv = len(os.listdir(f'{SSD_DATA}/csv_warehouse')) if os.path.isdir(f'{SSD_DATA}/csv_warehouse') else 0
n_meta = len(os.listdir(f'{SSD_DATA}/metadata')) if os.path.isdir(f'{SSD_DATA}/metadata') else 0
print(f'\n\u2705 Pipeline 1 hoàn tất: {n_csv} CSV, {n_meta} metadata files')

In [ ]:
# ============================================================
# Ô 5: NÉN DATA VÀ SAO LƯU VỀ DRIVE
# Chạy SAU KHI Ô 4 xong 100%. Lưu 1 file ZIP duy nhất lên Drive.
# Nick phụ sẽ giải nén file này để dùng lại data.
# ============================================================
import os

DRIVE_PROJECT = '/content/drive/MyDrive/Colab Notebooks/Text2Pandas'
ZIP_PATH = os.path.join(DRIVE_PROJECT, 'pipeline1_output.zip')

print('Đang nén data... (có thể mất 2-5 phút)')
!cd /content && zip -r -q "{ZIP_PATH}" data_output/

size_mb = os.path.getsize(ZIP_PATH) / 1024**2
print(f'\n\u2705 Đã lưu: {ZIP_PATH}')
print(f'   Dung lượng: {size_mb:.1f} MB')
print(f'   Nick phụ chỉ cần chạy Ô 5b để giải nén và dùng ngay!')

---
## 🚀 PIPELINE 2 (Nick chính hoặc nick phụ)

In [ ]:
# ============================================================
# Ô 5b: GIẢI NÉN DATA TỪ DRIVE (CHỈ DÙNG CHO NICK PHỤ)
# Nếu bạn vừa chạy xong Ô 4 + Ô 5 trên nick này rồi thì BỎ QUA ô này.
# ============================================================
import os

DRIVE_PROJECT = '/content/drive/MyDrive/Colab Notebooks/Text2Pandas'
ZIP_PATH = os.path.join(DRIVE_PROJECT, 'pipeline1_output.zip')

if os.path.exists(ZIP_PATH):
    print(f'\u0110ang giải nén {ZIP_PATH}...')
    !unzip -q -o "{ZIP_PATH}" -d /content/
    n_csv = len(os.listdir('/content/data_output/csv_warehouse'))
    n_meta = len(os.listdir('/content/data_output/metadata'))
    print(f'\n\u2705 Giải nén xong: {n_csv} CSV, {n_meta} metadata files')
else:
    print(f'\u274c Không tìm thấy {ZIP_PATH}')
    print('Hãy chạy Pipeline 1 (Ô 4 + Ô 5) trên nick chính trước!')

In [ ]:
# ============================================================
# Ô 6: PIPELINE 2 - QUERY + LLM (TỰ RESUME QUA CHECKPOINT)
# ============================================================
import os

DRIVE_DATA = '/content/drive/MyDrive/Colab Notebooks/Text2Pandas/data'
SSD_DATA   = '/content/data_output'
os.makedirs('/content/output', exist_ok=True)

!python -m src.pipeline_2_query.pipeline \
    --questions "{DRIVE_DATA}/test_questions.jsonl" \
    --code-stock "{DRIVE_DATA}/ViFinQA/code_stock.csv" \
    --metadata "{SSD_DATA}/metadata" \
    --csv-warehouse "{SSD_DATA}/csv_warehouse" \
    --index "{SSD_DATA}/index" \
    --output "/content/output/submission.json" \
    --checkpoint "/content/output/checkpoint_results.jsonl" \
    --use-llm --verbose

In [ ]:
# ============================================================
# Ô 7: KIỂM TRA KẾT QUẢ + SAO LƯU SUBMISSION VỀ DRIVE
# ============================================================
import json, os, shutil

chk = '/content/output/checkpoint_results.jsonl'
if os.path.exists(chk):
    with open(chk) as f:
        lines = [l for l in f if l.strip()]
    ok = sum(1 for l in lines if json.loads(l).get('success'))
    print(f'Checkpoint: {len(lines)} câu ({ok} thành công, {len(lines)-ok} thất bại)')

sub_path = '/content/output/submission.json'
if os.path.exists(sub_path):
    with open(sub_path) as f:
        sub = json.load(f)
    print(f'\nSubmission: {len(sub)} câu trả lời')
    for k, v in list(sub.items())[:5]:
        print(f'  Q{k}: {v}')
    
    DRIVE_OUT = '/content/drive/MyDrive/Colab Notebooks/Text2Pandas/output'
    os.makedirs(DRIVE_OUT, exist_ok=True)
    shutil.copy2(sub_path, os.path.join(DRIVE_OUT, 'submission.json'))
    print(f'\n\u2705 Đã sao lưu submission.json về Drive!')
else:
    print('Chưa có submission! Chạy Ô 6 trước.')